In [6]:
import csv, numpy as np
from CodonTransformer.CodonEvaluation import get_min_max_percentage

ModuleNotFoundError: No module named 'CodonTransformer'

In [ ]:
!  ../coconut -harmonize ../examples/codon_table_minmax_hsapiens ../examples/Mus_musculus_mm_table ../examples//example.fasta

In [3]:


GC = {'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
'TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
'AAT':'N','AAC':'N','AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E',
'TGT':'C','TGC':'C','TGA':'*','TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
'AGT':'S','AGC':'S','AGA':'R','AGG':'R','GGT':'G','GGC':'G','GGA':'G','GGG':'G'}
STOPS = {"TAA","TAG","TGA"}



In [4]:
def read_first(p):                      # first FASTA record only
    seq = []
    for l in open(p):
        if l.startswith(">"):
            if seq: break
            continue
        seq.append(l.strip())
    return "".join(seq).upper().replace("U", "T")

def translate(dna):
    return "".join(GC.get(dna[i:i+3], "X") for i in range(0, len(dna) - len(dna) % 3, 3))

orig = read_first("../examples/example.fasta")
harm = read_first("harmonized_cds.fasta")
oc = [orig[i:i+3] for i in range(0, len(orig) - len(orig) % 3, 3)]
hc = [harm[i:i+3] for i in range(0, len(harm) - len(harm) % 3, 3)]


In [5]:

# 1) CRITICAL: harmonized must encode the SAME protein (synonymous changes only)
print("same length       :", len(orig) == len(harm))
print("same protein      :", translate(orig) == translate(harm))
print("all changes synon :", all(GC[a] == GC[b] for a, b in zip(oc, hc)))
print("codons changed    :", sum(a != b for a, b in zip(oc, hc)), "/", len(oc))

# 2) Coconut's own fit summary (harmonized should be closest to native)
print("\nfit summary:")
for row in csv.DictReader(open("harmonize_fit_summary.csv")):
    print(" ", dict(row))

# 3) Independent: harmonized %MinMax (org2 table) should track native %MinMax (org1 table)
def load_cf(path):
    cf = {}
    for line in open(path):
        f = line.split()
        if len(f) < 10 or f[8].upper() == "END":       # sense codons only (END/End)
            continue
        cf.setdefault(f[8], ([], []))
        cf[f[8]][0].append(f[9].upper()); cf[f[8]][1].append(float(f[3]))
    return cf

def sense(dna):
    cod = [dna[i:i+3] for i in range(0, len(dna) - len(dna) % 3, 3)]
    return "".join(c for c in cod if c not in STOPS)



same length       : True
same protein      : True
all changes synon : True
codons changed    : 59 / 148

fit summary:
  {'ID': 'ENST00000335295.4', 'variant': 'native_in_org2', 'RMSE_vs_native': '10.202', 'max_abs_dev': '19.456', 'DTW_raw': '591.840', 'DTW_normalized': '3.13143'}
  {'ID': 'ENST00000335295.4', 'variant': 'maxexpr', 'RMSE_vs_native': '39.142', 'max_abs_dev': '68.069', 'DTW_raw': '4827.791', 'DTW_normalized': '36.85337'}
  {'ID': 'ENST00000335295.4', 'variant': 'harmonized', 'RMSE_vs_native': '2.913', 'max_abs_dev': '8.467', 'DTW_raw': '243.557', 'DTW_normalized': '1.60235'}


In [ ]:
cf1 = load_cf("../examples/codon_table_minmax_hsapiens")
cf2 = load_cf("../examples/Mus_musculus_mm_table")
native = np.array([v for v in get_min_max_percentage(sense(orig), cf1, 18) if v is not None])
harmon = np.array([v for v in get_min_max_percentage(sense(harm), cf2, 18) if v is not None])
n = min(len(native), len(harmon))
print("\nnative(org1) vs harmonized(org2) %MinMax:")
print("  Pearson r :", round(np.corrcoef(native[:n], harmon[:n])[0, 1], 4))
print("  max|Δ|    :", round(float(np.abs(native[:n] - harmon[:n]).max()), 2))